# Projet SM604 - De la classification des chiffres manuscrits à la détection de cancers du sein
## Mathématiques pour le Machine Learning - EFREI Paris 2025-2026
### Sabrina El Hassani, Aude Labat, Thomas Duriaud, Paul Fontaine, Evan Ladeira

---

## Partie 3 - Application au diagnostic médical (CBIS-DDSM)

Ce notebook est organisé en quatre sections :

**Section 3.1** Chargement et prétraitement du dataset CBIS-DDSM

**Section 3.2** Architectures denses (linéaire, H=1, H=2) adaptées à la classification binaire

**Section 3.3** CNN avec data augmentation et images 224×224

**Section 3.4** Analyse de la matrice de confusion et discussion médicale

---

### Présentation du dataset CBIS-DDSM

Le **CBIS-DDSM** (Curated Breast Imaging Subset of DDSM) est une version standardisée du DDSM,
base de données de référence en mammographie contenant 2 620 études cliniques avec pathologie vérifiée.
Les images DICOM ont été converties en JPEG et les annotations ROI ont été mises à jour par des
mammographes entraînés.

**Choix de travailler sur les ROI crops à 224×224 :**  
On utilise `cropped image file path` (ROI autour de la lésion) plutôt que la mammographie complète,
conformément aux recommandations du dataset. La résolution est fixée à **224×224** (vs 128×128
initialement) pour préserver les détails fins des lésions (spiculations, contours irréguliers)
qui sont les marqueurs diagnostiques clés.

**Data augmentation :** on multiplie le dataset d'entraînement par 4 via des transformations
géométriques médicalement valides (retournements horizontal et vertical, rotation ±15°)
pour réduire l'overfitting sévère observé avec seulement 1318 images.

**Objectif :** classification binaire **bénin (0) vs malin (1)**.

---
## Section 3.1 - Chargement et prétraitement de CBIS-DDSM

### Imports

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from PIL import Image
from sklearn.metrics import confusion_matrix

np.random.seed(42)
torch.manual_seed(42)
print("Imports OK")

### Chemins du dataset

**Deux chemins à adapter :**
- `CSV_DIR`  : dossier contenant les fichiers `.csv`
- `JPEG_DIR` : dossier contenant les sous-dossiers `1.3.6.1.4.1.9590.xxx/`

In [ ]:
# --- Adapter ces deux chemins ---
CSV_DIR  = "C:/Users/audel/Documents/csv"    # dossier contenant les .csv
JPEG_DIR = "C:/Users/audel/Documents/jpeg"   # dossier contenant les sous-dossiers 1.3.6...
IMG_SIZE = 224   # 224x224 pour préserver les détails fins des lésions

for d in [CSV_DIR, JPEG_DIR]:
    print(f"  {d} : {'OK' if os.path.isdir(d) else 'INTROUVABLE'}")

### Chargement des données via les ROI crops

**Stratégie de chargement :**  
On utilise `cropped image file path` (ROI crop autour de la lésion).  
Le 3ème segment du chemin DICOM = `SeriesInstanceUID` que l'on retrouve dans `dicom_info.csv`  
avec `SeriesDescription = 'cropped images'` pour obtenir le chemin `.jpg` réel.

**Note :** pas de déduplication — 1 lésion = 1 crop unique = 1 exemple.

In [ ]:
def charger_cbis_roi(csv_masse, csv_dicom, jpeg_dir, taille=224):
    """
    Charge les ROI crops du dataset CBIS-DDSM.
    Utilise 'cropped image file path' plutot que 'image file path'
    pour concentrer le signal sur la lesion elle-meme.

    Strategie :
    1. Lire le CSV masse (pathology + chemin DICOM du crop)
    2. Extraire le SeriesInstanceUID (3e segment du chemin DICOM crop)
    3. Joindre avec dicom_info.csv (SeriesDescription='cropped images')
    4. Charger chaque image, convertir en niveaux de gris, redimensionner

    csv_masse : str, chemin vers mass_case_description_train/test_set.csv
    csv_dicom : str, chemin vers dicom_info.csv
    jpeg_dir  : str, dossier racine des images JPEG
    taille    : int, cote du carre cible (224x224)
    Retourne  : X array (n, taille, taille) float32, y array (n,) int
    """
    # Etape 1 : lecture des CSV
    df_masse = pd.read_csv(csv_masse)
    df_dicom = pd.read_csv(csv_dicom)

    # Etape 2 : binarisation de la pathologie
    def label(p):
        p = str(p).strip().upper()
        if p in ["BENIGN", "BENIGN_WITHOUT_CALLBACK"]: return 0
        if p == "MALIGNANT": return 1
        return -1

    df_masse["label"] = df_masse["pathology"].apply(label)
    df_masse = df_masse[df_masse["label"] != -1].reset_index(drop=True)

    # Etape 3 : extraction du SeriesInstanceUID depuis le chemin du crop
    # chemin DICOM = 'Mass-Training_P_XXXXX_1 / UID_serie / UID_instance / 000000.dcm'
    # Le 3e segment (index 2) = UID_instance = SeriesInstanceUID dans dicom_info
    df_masse["UID_crop"] = df_masse["cropped image file path"].apply(
        lambda p: p.replace("\\", "/").split("/")[2]
    )

    # Etape 4 : jointure avec dicom_info sur les crops uniquement
    df_crops = df_dicom[
        df_dicom["SeriesDescription"].str.contains("cropped", case=False, na=False)
    ][["SeriesInstanceUID", "image_path"]].drop_duplicates("SeriesInstanceUID")

    df_masse = df_masse.merge(
        df_crops, left_on="UID_crop", right_on="SeriesInstanceUID", how="left"
    )
    df_masse = df_masse[df_masse["image_path"].notna()].reset_index(drop=True)

    # image_path = 'CBIS-DDSM/jpeg/UID/2-249.jpg'
    # -> on extrait 'UID/2-249.jpg' et on prefixe avec jpeg_dir
    df_masse["chemin_jpeg"] = df_masse["image_path"].apply(
        lambda p: os.path.join(jpeg_dir, "/".join(str(p).replace("\\", "/").split("/")[-2:]))
    )

    # Pas de deduplication : 1 lesion = 1 crop = 1 exemple
    print(f"  {len(df_masse)} ROI crops | "
          f"Benin={int((df_masse['label']==0).sum())}  "
          f"Malin={int((df_masse['label']==1).sum())}")

    # Etape 5 : chargement des images
    images, labels = [], []
    n_manquantes   = 0

    for i, row in df_masse.iterrows():
        try:
            img = Image.open(row["chemin_jpeg"]).convert("L")       # niveaux de gris
            img = img.resize((taille, taille), Image.BICUBIC)       # 224x224
            images.append(np.array(img, dtype=np.float32) / 255.0)  # normalisation [0,1]
            labels.append(row["label"])
        except Exception:
            n_manquantes += 1

        if (i + 1) % 100 == 0:
            print(f"    {len(images)} chargees...", end="\r")

    if n_manquantes > 0:
        print(f"    {n_manquantes} fichiers introuvables (verifier JPEG_DIR)")

    return np.array(images, dtype=np.float32), np.array(labels, dtype=int)


print("Chargement du train (ROI crops 224x224)...")
X_raw_train, y_train = charger_cbis_roi(
    os.path.join(CSV_DIR, "mass_case_description_train_set.csv"),
    os.path.join(CSV_DIR, "dicom_info.csv"),
    JPEG_DIR, taille=IMG_SIZE
)

print("\nChargement du test (ROI crops 224x224)...")
X_raw_test, y_test = charger_cbis_roi(
    os.path.join(CSV_DIR, "mass_case_description_test_set.csv"),
    os.path.join(CSV_DIR, "dicom_info.csv"),
    JPEG_DIR, taille=IMG_SIZE
)

print(f"\nTrain : {X_raw_train.shape}  |  Test : {X_raw_test.shape}")
print(f"Pixels : min={X_raw_train.min():.3f}  max={X_raw_train.max():.3f}")

### Visualisation des ROI crops

On affiche quelques crops de chaque classe à 224×224.  
La résolution plus élevée permet de mieux distinguer les textures caractéristiques :
contours lobulés pour les bénins, spiculations irrégulières pour les malins.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for classe, titre, couleur in [(0, "Benin", "green"), (1, "Malin", "red")]:
    indices_classe = np.where(y_train == classe)[0]
    echantillon    = np.random.choice(indices_classe, size=5, replace=False)
    for j, idx in enumerate(echantillon):
        ax = axes[classe][j]
        ax.imshow(X_raw_train[idx], cmap="gray", vmin=0, vmax=1)
        ax.set_title(titre, fontsize=10, color=couleur)
        ax.axis("off")

plt.suptitle("CBIS-DDSM - ROI crops de masses mammaires 224x224", fontsize=13)
plt.tight_layout()
plt.savefig("cbis_roi_echantillon.png", dpi=100, bbox_inches="tight")
plt.show()

### Préparation des données pour les modèles denses

On aplatit chaque crop 224×224 en vecteur $\vec{x} \in \mathbb{R}^{50176}$ (car $224 \times 224 = 50\,176$).  
On calcule les poids de déséquilibre pour la `CrossEntropyLoss` pondérée du CNN :

$$w_c = \frac{n_{\text{total}}}{2 \cdot n_c}$$

In [ ]:
N_PIXELS = IMG_SIZE * IMG_SIZE   # 50 176 pour 224x224

def one_hot(y, C=2):
    """
    Convertit les etiquettes entieres en matrice one-hot.
    y=0 donne [1, 0]  (benin)
    y=1 donne [0, 1]  (malin)

    y : array (n,) entiers entre 0 et C-1
    Retourne Y : array (n, C)
    """
    n = len(y)
    Y = np.zeros((n, C))
    Y[np.arange(n), y] = 1.0
    return Y


# Aplatissement : (n, 224, 224) -> (n, 50176)
X_train_flat = X_raw_train.reshape(len(X_raw_train), -1)
X_test_flat  = X_raw_test.reshape(len(X_raw_test),  -1)

# One-hot binaire C=2
Y_train_oh = one_hot(y_train, C=2)
Y_test_oh  = one_hot(y_test,  C=2)

# Poids de desequilibre pour la CrossEntropyLoss ponderee (CNN)
n_total_tr = len(y_train)
n_benin_tr = int((y_train == 0).sum())
n_malin_tr = int((y_train == 1).sum())
w_benin    = n_total_tr / (2 * n_benin_tr)
w_malin    = n_total_tr / (2 * n_malin_tr)

print(f"X_train_flat : {X_train_flat.shape}")
print(f"X_test_flat  : {X_test_flat.shape}")
print(f"\nDistribution train : Benin={n_benin_tr}  Malin={n_malin_tr}")
print(f"Poids de classe    : w_benin={w_benin:.3f}  w_malin={w_malin:.3f}")
print(f"Rapport            : une erreur sur un malin penalisee {w_malin/w_benin:.1f}x plus")

class_weights_torch = torch.tensor([w_benin, w_malin], dtype=torch.float32)

---
## Section 3.2 - Architectures denses adaptées à la classification binaire

On réutilise exactement les mêmes architectures que les Parties 1 et 2, avec deux adaptations :  
- **C = 2** classes au lieu de 10  
- **n_input = 50 176** au lieu de 784 (MNIST) ou 1024/3072 (CIFAR-10)

### Fonctions communes (softmax, cross-entropy, ReLU)

In [ ]:
def softmax_python(o):
    """
    Transforme les scores bruts en probabilites.
    Soustrait le max avant l'exponentielle pour la stabilite numerique.

    o : array (n, C)
    Retourne P : array (n, C), chaque ligne somme a 1
    """
    o = np.array(o)
    resultats = []
    for image in o:
        max_score      = max(image)
        scores_stables = [s - max_score for s in image]
        exp_scores     = [math.exp(s) for s in scores_stables]
        somme          = sum(exp_scores)
        probas         = [e / somme for e in exp_scores]
        resultats.append(probas)
    return np.array(resultats)


def cross_entropy(P, Y):
    """
    Mesure l'erreur moyenne entre probabilites predites et vraies etiquettes.
    Grace au one-hot, seul le log de la vraie classe compte par image.

    P : (n, C) probabilites predites
    Y : (n, C) etiquettes one-hot
    Retourne un scalaire
    """
    n = len(P)
    C = Y.shape[1]
    total = 0
    for i in range(n):
        for k in range(C):
            if Y[i][k] == 1:
                p = max(P[i][k], 1e-12)
                total += math.log(p)
                break
    return -total / n


def error_rate(P, y_true):
    """
    Proportion d'images mal classees.

    P      : (n, C) probabilites predites
    y_true : (n,) etiquettes entieres
    Retourne un scalaire entre 0 et 1
    """
    n = len(y_true)
    erreurs = 0
    for i in range(n):
        if int(np.argmax(P[i])) != y_true[i]:
            erreurs += 1
    return erreurs / n


def relu(x):
    """
    Applique ReLU element par element : max(0, x).
    Met les valeurs negatives a 0, garde les positives.
    Sans cette non-linearite, deux couches lineaires s'effondrent en une seule.

    x : array de shape quelconque
    Retourne un array de meme shape
    """
    return np.maximum(0, x)


def relu_derivative(x):
    """
    Derivee de ReLU : 1 si x > 0, 0 sinon.
    Utilisee dans la retropropagation pour bloquer le gradient
    la ou ReLU a eteint le neurone.

    x : array de shape quelconque
    Retourne un array de meme shape, valeurs 0 ou 1
    """
    return (x > 0).astype(float)


print("Fonctions communes chargees.")

### Modèle linéaire

Architecture : $\vec{o} = A\vec{x} + b$ avec $A \in \mathbb{R}^{2 \times 50176}$, $b \in \mathbb{R}^{2}$.  
Nombre de paramètres : $2 \times 50176 + 2 = 100\,354$.

**Gradient :**
$$\nabla_A L = \frac{1}{n}(P-Y)^T X \qquad \nabla_b L = \frac{1}{n}\sum_{\text{lignes}}(P-Y)$$

In [ ]:
def train_linear(X_train, Y_train, y_train, X_test, y_test,
                 n_input, n_classes=2, lr=0.01, batch_size=32, seuil=1e-4, max_epochs=500):
    """
    Descente de gradient par mini-batches avec early stopping.

    n_input    : dimension de l'entree (50176 pour 224x224)
    n_classes  : nombre de classes (2 pour classification binaire)
    lr         : learning rate eta
    batch_size : images par mini-batch
    seuil      : arret si |loss[t] - loss[t-1]| < seuil
    max_epochs : securite contre boucle infinie
    """
    A = np.random.randn(n_classes, n_input) * 0.01
    b = np.zeros(n_classes)
    n = len(X_train)
    epoch    = 0
    converge = False

    hist_loss, hist_train, hist_test = [], [], []

    while not converge:

        # Melange : toutes les images vues dans un ordre different a chaque epoch
        indices = np.random.permutation(n)
        X_s = X_train[indices]
        Y_s = Y_train[indices]

        for start in range(0, n, batch_size):
            X_b = X_s[start : start+batch_size]
            Y_b = Y_s[start : start+batch_size]

            # Passe avant
            o   = X_b @ A.T + b
            P_b = softmax_python(o)

            # Gradient : delta = P - Y
            delta = P_b - Y_b
            dA    = (delta.T @ X_b) / len(X_b)
            db    = delta.mean(axis=0)

            A -= lr * dA
            b -= lr * db

        # Evaluation fin d'epoch
        P_train = softmax_python(X_train @ A.T + b)
        P_test  = softmax_python(X_test  @ A.T + b)

        loss      = cross_entropy(P_train, Y_train)
        err_train = error_rate(P_train, y_train)
        err_test  = error_rate(P_test,  y_test)

        hist_loss.append(loss)
        hist_train.append(err_train)
        hist_test.append(err_test)

        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d} | Loss={loss:.4f} | "
                  f"Train={err_train*100:.2f}% | Test={err_test*100:.2f}%")

        if epoch > 0:
            diff = abs(hist_loss[-2] - hist_loss[-1])
            if diff < seuil:
                print(f"\nConverge a l'epoch {epoch} (diff={diff:.8f})")
                converge = True
            elif epoch > 5:
                if all(hist_test[-i] > hist_test[-i-1] for i in range(1, 5)):
                    print(f"\nEarly stopping epoch {epoch} : overfitting detecte")
                    converge = True

        if epoch >= max_epochs:
            print(f"\nArret : {max_epochs} epochs atteintes")
            converge = True

        epoch += 1

    return A, b, hist_loss, hist_train, hist_test

In [ ]:
np.random.seed(42)
print("=" * 55)
print(f"Modele lineaire - CBIS-DDSM | Parametres : {2*N_PIXELS + 2:,}")
print("=" * 55)

A_lin, b_lin, losses_lin, etr_lin, ete_lin = train_linear(
    X_train_flat, Y_train_oh, y_train,
    X_test_flat,  y_test,
    n_input=N_PIXELS, n_classes=2, lr=0.01, batch_size=32, seuil=1e-4, max_epochs=50
)

print(f"\nResultat final :")
print(f"  Train : {etr_lin[-1]*100:.2f}%  |  Test : {ete_lin[-1]*100:.2f}%  |  Ecart : {abs(etr_lin[-1]-ete_lin[-1])*100:.2f}%")

### Modèle H=1

Architecture : une couche cachée de 128 neurones avec ReLU.  
Nombre de paramètres : $128 \times 50176 + 128 + 2 \times 128 + 2 = 6\,423\,298$.

**Rétropropagation H=1 :**
$$\delta^2 = P - Y \qquad \nabla_{A^2} L = \frac{1}{n} (\delta^2)^T z^1 \qquad \nabla_{b^2} L = \frac{1}{n}\sum \delta^2$$
$$\delta^1 = (\delta^2 \cdot A^2) \times \text{ReLU}'(o^1) \qquad \nabla_{A^1} L = \frac{1}{n} (\delta^1)^T X \qquad \nabla_{b^1} L = \frac{1}{n}\sum \delta^1$$

In [ ]:
def train_h1(X_train, Y_train, y_train, X_test, y_test,
             n_input, p1=128, n_classes=2, lr=0.01, batch_size=32, seuil=1e-4, max_epochs=500):
    """
    Entrainement du modele H=1 par descente de gradient mini-batches.

    n_input    : dimension de l'entree
    p1         : nombre de neurones dans la couche cachee
    n_classes  : nombre de classes
    lr         : learning rate
    batch_size : images par mini-batch
    seuil      : arret si |loss[t] - loss[t-1]| < seuil
    max_epochs : securite contre boucle infinie
    """
    A1 = np.random.randn(p1, n_input)   * 0.01
    b1 = np.zeros(p1)
    A2 = np.random.randn(n_classes, p1) * 0.01
    b2 = np.zeros(n_classes)
    n  = len(X_train)
    epoch    = 0
    converge = False

    hist_loss, hist_train, hist_test = [], [], []

    while not converge:

        indices = np.random.permutation(n)
        X_s = X_train[indices]
        Y_s = Y_train[indices]

        for start in range(0, n, batch_size):
            X_b = X_s[start : start+batch_size]
            Y_b = Y_s[start : start+batch_size]

            # Forward
            o1  = X_b @ A1.T + b1
            z1  = relu(o1)
            o2  = z1  @ A2.T + b2
            P_b = softmax_python(o2)

            # Backward
            delta2 = P_b - Y_b
            dA2    = (delta2.T @ z1) / len(X_b)
            db2    = delta2.mean(axis=0)
            delta1 = (delta2 @ A2) * relu_derivative(o1)
            dA1    = (delta1.T @ X_b) / len(X_b)
            db1    = delta1.mean(axis=0)

            A1 -= lr * dA1
            b1 -= lr * db1
            A2 -= lr * dA2
            b2 -= lr * db2

        # Evaluation fin d'epoch
        z1_tr   = relu(X_train @ A1.T + b1)
        P_train = softmax_python(z1_tr @ A2.T + b2)
        z1_te   = relu(X_test  @ A1.T + b1)
        P_test  = softmax_python(z1_te @ A2.T + b2)

        loss      = cross_entropy(P_train, Y_train)
        err_train = error_rate(P_train, y_train)
        err_test  = error_rate(P_test,  y_test)

        hist_loss.append(loss)
        hist_train.append(err_train)
        hist_test.append(err_test)

        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d} | Loss={loss:.4f} | "
                  f"Train={err_train*100:.2f}% | Test={err_test*100:.2f}%")

        if epoch > 0:
            diff = abs(hist_loss[-2] - hist_loss[-1])
            if diff < seuil:
                print(f"\nConverge a l'epoch {epoch} (diff={diff:.8f})")
                converge = True
            elif epoch > 5:
                if all(hist_test[-i] > hist_test[-i-1] for i in range(1, 5)):
                    print(f"\nEarly stopping epoch {epoch} : overfitting detecte")
                    converge = True

        if epoch >= max_epochs:
            print(f"\nArret : {max_epochs} epochs atteintes")
            converge = True

        epoch += 1

    return A1, b1, A2, b2, hist_loss, hist_train, hist_test

In [ ]:
np.random.seed(42)
print("=" * 55)
print(f"Modele H=1 - CBIS-DDSM (p1=128) | Parametres : {128*N_PIXELS+128+2*128+2:,}")
print("=" * 55)

A1_h1, b1_h1, A2_h1, b2_h1, losses_h1, etr_h1, ete_h1 = train_h1(
    X_train_flat, Y_train_oh, y_train,
    X_test_flat,  y_test,
    n_input=N_PIXELS, p1=128, n_classes=2, lr=0.01, batch_size=32, seuil=1e-4, max_epochs=50
)

print(f"\nResultat final :")
print(f"  Train : {etr_h1[-1]*100:.2f}%  |  Test : {ete_h1[-1]*100:.2f}%  |  Ecart : {abs(etr_h1[-1]-ete_h1[-1])*100:.2f}%")

### Modèle H=2

Architecture : deux couches cachées (128 puis 64 neurones) avec ReLU.  
Nombre de paramètres : $128 \times 50176 + 128 + 64 \times 128 + 64 + 2 \times 64 + 2 = 6\,431\,554$.

In [ ]:
def train_h2(X_train, Y_train, y_train, X_test, y_test,
             n_input, p1=128, p2=64, n_classes=2, lr=0.01, batch_size=32, seuil=1e-4, max_epochs=500):
    """
    Entrainement du modele H=2 par descente de gradient mini-batches.

    n_input    : dimension de l'entree
    p1, p2     : nombre de neurones dans les deux couches cachees
    n_classes  : nombre de classes
    lr         : learning rate
    batch_size : images par mini-batch
    seuil      : arret si |loss[t] - loss[t-1]| < seuil
    max_epochs : securite contre boucle infinie
    """
    A1 = np.random.randn(p1, n_input)   * 0.01
    b1 = np.zeros(p1)
    A2 = np.random.randn(p2, p1)        * 0.01
    b2 = np.zeros(p2)
    A3 = np.random.randn(n_classes, p2) * 0.01
    b3 = np.zeros(n_classes)
    n  = len(X_train)
    epoch    = 0
    converge = False

    hist_loss, hist_train, hist_test = [], [], []

    while not converge:

        indices = np.random.permutation(n)
        X_s = X_train[indices]
        Y_s = Y_train[indices]

        for start in range(0, n, batch_size):
            X_b = X_s[start : start+batch_size]
            Y_b = Y_s[start : start+batch_size]

            # Forward
            o1  = X_b @ A1.T + b1
            z1  = relu(o1)
            o2  = z1  @ A2.T + b2
            z2  = relu(o2)
            o3  = z2  @ A3.T + b3
            P_b = softmax_python(o3)

            # Backward
            delta3 = P_b - Y_b
            dA3    = (delta3.T @ z2) / len(X_b)
            db3    = delta3.mean(axis=0)
            delta2 = (delta3 @ A3) * relu_derivative(o2)
            dA2    = (delta2.T @ z1) / len(X_b)
            db2    = delta2.mean(axis=0)
            delta1 = (delta2 @ A2) * relu_derivative(o1)
            dA1    = (delta1.T @ X_b) / len(X_b)
            db1    = delta1.mean(axis=0)

            A1 -= lr * dA1
            b1 -= lr * db1
            A2 -= lr * dA2
            b2 -= lr * db2
            A3 -= lr * dA3
            b3 -= lr * db3

        # Evaluation fin d'epoch
        z1_tr   = relu(X_train @ A1.T + b1)
        z2_tr   = relu(z1_tr   @ A2.T + b2)
        P_train = softmax_python(z2_tr @ A3.T + b3)
        z1_te   = relu(X_test  @ A1.T + b1)
        z2_te   = relu(z1_te   @ A2.T + b2)
        P_test  = softmax_python(z2_te @ A3.T + b3)

        loss      = cross_entropy(P_train, Y_train)
        err_train = error_rate(P_train, y_train)
        err_test  = error_rate(P_test,  y_test)

        hist_loss.append(loss)
        hist_train.append(err_train)
        hist_test.append(err_test)

        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d} | Loss={loss:.4f} | "
                  f"Train={err_train*100:.2f}% | Test={err_test*100:.2f}%")

        if epoch > 0:
            diff = abs(hist_loss[-2] - hist_loss[-1])
            if diff < seuil:
                print(f"\nConverge a l'epoch {epoch} (diff={diff:.8f})")
                converge = True
            elif epoch > 5:
                if all(hist_test[-i] > hist_test[-i-1] for i in range(1, 5)):
                    print(f"\nEarly stopping epoch {epoch} : overfitting detecte")
                    converge = True

        if epoch >= max_epochs:
            print(f"\nArret : {max_epochs} epochs atteintes")
            converge = True

        epoch += 1

    return A1, b1, A2, b2, A3, b3, hist_loss, hist_train, hist_test

In [ ]:
np.random.seed(42)
print("=" * 60)
print(f"Modele H=2 - CBIS-DDSM (p1=128, p2=64) | Parametres : {128*N_PIXELS+128+64*128+64+2*64+2:,}")
print("=" * 60)

A1_h2, b1_h2, A2_h2, b2_h2, A3_h2, b3_h2, losses_h2, etr_h2, ete_h2 = train_h2(
    X_train_flat, Y_train_oh, y_train,
    X_test_flat,  y_test,
    n_input=N_PIXELS, p1=128, p2=64, n_classes=2, lr=0.01, batch_size=32, seuil=1e-4, max_epochs=50
)

print(f"\nResultat final :")
print(f"  Train : {etr_h2[-1]*100:.2f}%  |  Test : {ete_h2[-1]*100:.2f}%  |  Ecart : {abs(etr_h2[-1]-ete_h2[-1])*100:.2f}%")

In [ ]:
# Courbes d'erreur train/test pour les 3 architectures denses
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titres  = ["Lineaire", "H=1 (128)", "H=2 (128, 64)"]
donnees = [(etr_lin, ete_lin), (etr_h1, ete_h1), (etr_h2, ete_h2)]

for ax, titre, (etr, ete) in zip(axes, titres, donnees):
    ax.plot([e*100 for e in etr], label="Train", color="steelblue")
    ax.plot([e*100 for e in ete], label="Test",  color="coral")
    ax.set_title(titre)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Erreur (%)")
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle("CBIS-DDSM ROI 224x224 - Architectures denses (benin vs malin)", fontsize=13)
plt.tight_layout()
plt.savefig("courbes_cbis_dense.png", dpi=100, bbox_inches="tight")
plt.show()

---
## Section 3.3 - CNN avec data augmentation et images 224×224

**Deux améliorations par rapport au CNN de base :**

**1. Images 224×224** : les ROI crops contiennent des détails fins (spiculations, contours irréguliers)
qui disparaissent à 128×128. À 224×224, ces structures restent visibles et discriminantes.
Après 3 max-poolings : $224 \to 112 \to 56 \to 28$, donc flatten = $28 \times 28 \times 64 = 50\,176$.

**2. Data augmentation** : on multiplie le dataset train par 4 via trois transformations
géométriques médicalement valides sur des mammographies :
- **Retournement horizontal** : gauche/droite sont symétriques en mammographie
- **Retournement vertical** : haut/bas acceptable sur des ROI crops
- **Rotation +15°** : la lésion reste visible et cliniquement plausible

On ne fait **pas** de zoom ni de déformation élastique : ces transformations
pourraient altérer les contours de la masse qui sont précisément les marqueurs diagnostiques.

In [ ]:
class CNN_CBIS(nn.Module):
    """
    CNN pour la classification binaire de ROI crops CBIS-DDSM a 224x224.
    Entree  : (batch, 1, 224, 224) ROI crop en niveaux de gris normalisee
    Sortie  : (batch, 2) logits bruts

    Architecture :
    - Bloc 1 : Conv(1->32) + Conv(32->32) + MaxPool -> (32, 112, 112)
    - Bloc 2 : Conv(32->64) + Conv(64->64) + MaxPool -> (64, 56, 56)
    - Bloc 3 : Conv(64->64) + MaxPool -> (64, 28, 28)
    - Flatten : 64x28x28 = 50 176 valeurs
    - Dropout(0.5) + Linear(50176 -> 2)
    """
    def __init__(self):
        super(CNN_CBIS, self).__init__()

        # Bloc 1 : 1 canal gris en entree, 32 cartes de caracteristiques
        self.conv1 = nn.Conv2d(in_channels=1,  out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)   # 224x224 -> 112x112

        # Bloc 2 : 32 -> 64 cartes, capture des motifs plus abstraits
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)   # 112x112 -> 56x56

        # Bloc 3 : 64 cartes supplementaires pour les textures fines des lesions
        self.conv5 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)   # 56x56 -> 28x28

        # Tete de classification : 64x28x28 = 50176 -> 2 classes
        self.dropout = nn.Dropout(p=0.5)           # regularisation contre l'overfitting
        self.fc      = nn.Linear(64 * 28 * 28, 2)
        self.relu    = nn.ReLU()

    def forward(self, x):
        """
        Passage avant : chemin des donnees a travers les couches.
        x entree : (batch, 1, 224, 224)
        """
        x = self.relu(self.conv1(x))   # (batch, 32, 224, 224)
        x = self.relu(self.conv2(x))   # (batch, 32, 224, 224)
        x = self.pool1(x)              # (batch, 32, 112, 112)

        x = self.relu(self.conv3(x))   # (batch, 64, 112, 112)
        x = self.relu(self.conv4(x))   # (batch, 64, 112, 112)
        x = self.pool2(x)              # (batch, 64, 56, 56)

        x = self.relu(self.conv5(x))   # (batch, 64, 56, 56)
        x = self.pool3(x)              # (batch, 64, 28, 28)

        x = torch.flatten(x, 1)        # (batch, 50176)
        x = self.dropout(x)            # eteint 50% des neurones pendant l'entrainement
        x = self.fc(x)                 # (batch, 2) logits bruts
        return x


# Verification de l'architecture
model_cnn = CNN_CBIS()
n_params  = sum(p.numel() for p in model_cnn.parameters() if p.requires_grad)
print(model_cnn)
print(f"\nTotal parametres entrainables : {n_params:,}")

# Test avec un batch fictif 224x224
x_fictif = torch.randn(4, 1, 224, 224)
sortie   = model_cnn(x_fictif)
print(f"Test forward : entree {list(x_fictif.shape)} -> sortie {list(sortie.shape)}")

### Préparation des données PyTorch et data augmentation

La data augmentation est appliquée **uniquement sur le train**, jamais sur le test.  
Le test doit rester identique aux conditions réelles pour évaluer la vraie généralisation.

Chaque image originale génère 3 copies → dataset train $\times 4$.

In [ ]:
# Ajout dimension canal : (n, 224, 224) -> (n, 1, 224, 224)
X_tr_base = torch.tensor(X_raw_train[:, np.newaxis, :, :], dtype=torch.float32)
X_te_t    = torch.tensor(X_raw_test[:,  np.newaxis, :, :], dtype=torch.float32)
y_tr_base = torch.tensor(y_train, dtype=torch.long)
y_te_t    = torch.tensor(y_test,  dtype=torch.long)

# Normalisation par la moyenne et l'ecart-type du train
# On applique les memes stats au test pour eviter la fuite d'information
mean_cbis = X_tr_base.mean()
std_cbis  = X_tr_base.std()
X_tr_base = (X_tr_base - mean_cbis) / (std_cbis + 1e-8)
X_te_t    = (X_te_t    - mean_cbis) / (std_cbis + 1e-8)

print(f"X_tr_base : {list(X_tr_base.shape)}")
print(f"X_te_t    : {list(X_te_t.shape)}")
print(f"Normalisation : mean={mean_cbis:.4f}  std={std_cbis:.4f}")


def augmenter(X, y):
    """
    Augmente le dataset d'entrainement par transformations geometriques.
    Chaque image originale genere 3 copies -> dataset x4.

    Transformations choisies (medicalement valides sur des mammographies) :
    - Retournement horizontal : gauche/droite symetriques
    - Retournement vertical   : haut/bas acceptable sur des ROI crops
    - Rotation +15 degres     : lesion reste visible et plausible

    Transformations exclues :
    - Zoom / deformation : pourrait alterer les contours diagnostiques

    X : Tensor (n, 1, H, W)
    y : Tensor (n,)
    Retourne X_aug (4n, 1, H, W), y_aug (4n,)
    """
    # Retournement horizontal (miroir gauche/droite)
    X_flip_h = torch.flip(X, dims=[3])

    # Retournement vertical (miroir haut/bas)
    X_flip_v = torch.flip(X, dims=[2])

    # Rotation de +15 degres via matrice affine
    # On utilise F.affine_grid + F.grid_sample pour appliquer la rotation
    angle = 15 * math.pi / 180   # conversion degres -> radians
    cos_a = math.cos(angle)
    sin_a = math.sin(angle)

    # Matrice de rotation 2x3 pour F.affine_grid
    theta = torch.tensor([[cos_a, -sin_a, 0],
                           [sin_a,  cos_a, 0]], dtype=torch.float32)
    theta = theta.unsqueeze(0).expand(len(X), -1, -1)     # (n, 2, 3)
    grid  = F.affine_grid(theta, X.size(), align_corners=False)
    # padding_mode='reflection' : les bords sont remplis par reflexion,
    # ce qui evite les artefacts noirs aux coins apres rotation
    X_rot = F.grid_sample(X, grid, align_corners=False, padding_mode="reflection")

    # Concatenation : original + 3 augmentations
    X_aug = torch.cat([X, X_flip_h, X_flip_v, X_rot], dim=0)   # (4n, 1, H, W)
    y_aug = torch.cat([y, y,        y,        y],       dim=0)  # (4n,)

    # Melange pour eviter que les 4 copies d'une meme image soient groupees
    idx = torch.randperm(len(X_aug))
    return X_aug[idx], y_aug[idx]


X_tr_aug, y_tr_aug = augmenter(X_tr_base, y_tr_base)

print(f"\nApres augmentation :")
print(f"  Train original : {len(X_tr_base)} images")
print(f"  Train augmente : {len(X_tr_aug)} images (x4)")
print(f"  Test           : {len(X_te_t)} images (inchange)")

n_benin_aug = int((y_tr_aug == 0).sum())
n_malin_aug = int((y_tr_aug == 1).sum())
print(f"\nDistribution apres augmentation : Benin={n_benin_aug}  Malin={n_malin_aug}")

# DataLoader avec le dataset augmente
train_loader = DataLoader(
    TensorDataset(X_tr_aug, y_tr_aug), batch_size=32, shuffle=True
)
test_loader  = DataLoader(
    TensorDataset(X_te_t, y_te_t), batch_size=64, shuffle=False
)

print(f"\nBatches entrainement : {len(train_loader)}  (sur {len(X_tr_aug)} images augmentees)")
print(f"Batches test          : {len(test_loader)}  (sur {len(X_te_t)} images originales)")

### Entraînement du CNN médical

On suit la **sensibilité** (rappel sur la classe maligne) à chaque epoch :

$$\text{Sensibilite} = \frac{VP}{VP + FN}$$

où $VP$ = vrais positifs et $FN$ = faux négatifs (cancers manqués).  
C'est la métrique prioritaire en diagnostic.

In [ ]:
def train_cnn_medical(model, train_loader, test_loader, class_weights, n_epochs=30, lr=1e-3):
    """
    Entrainement du CNN avec CrossEntropyLoss ponderee.
    Suit le taux d'erreur global ET la sensibilite (rappel malin) a chaque epoch.

    model         : instance de CNN_CBIS
    train_loader  : DataLoader entrainement (avec data augmentation)
    test_loader   : DataLoader test (images originales uniquement)
    class_weights : Tensor(2,) poids inversement proportionnels aux frequences
    n_epochs      : nombre d'epochs
    lr            : learning rate
    """
    criterion = nn.CrossEntropyLoss(weight=class_weights)   # penalise plus les erreurs sur les malins
    optimizer = optim.Adam(model.parameters(), lr=lr)

    hist_loss_tr, hist_loss_te = [], []
    hist_err_tr,  hist_err_te  = [], []
    hist_sensib                = []

    for epoch in range(n_epochs):

        # Phase d'entrainement
        model.train()                          # active le mode entrainement (dropout actif)
        loss_cumul, erreurs_tr = 0, 0

        for Xb, yb in train_loader:
            optimizer.zero_grad()              # reinitialise les gradients accumules
            sortie = model(Xb)                 # passe avant : (batch, 2) logits
            loss   = criterion(sortie, yb)     # calcul de la loss ponderee
            loss.backward()                    # retropropagation
            optimizer.step()                   # mise a jour des poids
            loss_cumul  += loss.item() * len(Xb)
            erreurs_tr  += int((sortie.argmax(dim=1) != yb).sum())

        loss_tr = loss_cumul / len(train_loader.dataset)
        err_tr  = erreurs_tr / len(train_loader.dataset)

        # Phase d'evaluation
        model.eval()                           # desactive le dropout
        loss_cumul, erreurs_te = 0, 0
        vp_malin, fn_malin     = 0, 0

        with torch.no_grad():
            for Xb, yb in test_loader:
                sortie      = model(Xb)
                preds       = sortie.argmax(dim=1)
                loss_cumul += criterion(sortie, yb).item() * len(Xb)
                erreurs_te += int((preds != yb).sum())
                mask         = (yb == 1)
                vp_malin    += int((preds[mask] == 1).sum())
                fn_malin    += int((preds[mask] == 0).sum())

        loss_te = loss_cumul / len(test_loader.dataset)
        err_te  = erreurs_te / len(test_loader.dataset)
        sensib  = vp_malin / (vp_malin + fn_malin + 1e-8)

        hist_loss_tr.append(loss_tr)
        hist_loss_te.append(loss_te)
        hist_err_tr.append(err_tr)
        hist_err_te.append(err_te)
        hist_sensib.append(sensib)

        print(f"  Epoch {epoch+1:2d}/{n_epochs} | "
              f"Loss tr={loss_tr:.4f} te={loss_te:.4f} | "
              f"Erreur tr={err_tr*100:.2f}% te={err_te*100:.2f}% | "
              f"Sensibilite={sensib*100:.1f}%")

    return hist_loss_tr, hist_loss_te, hist_err_tr, hist_err_te, hist_sensib


print("train_cnn_medical chargee.")

In [ ]:
torch.manual_seed(42)
model_cnn = CNN_CBIS()

print("=" * 70)
print(f"CNN CBIS-DDSM 224x224 + augmentation | Parametres : {sum(p.numel() for p in model_cnn.parameters()):,}")
print(f"Dataset train augmente : {len(X_tr_aug)} images (x4 original)")
print("Note : sur CPU environ 30-60 min pour 30 epochs (images plus grandes).")
print("=" * 70)

hist_loss_tr, hist_loss_te, hist_err_tr, hist_err_te, hist_sensib = train_cnn_medical(
    model_cnn, train_loader, test_loader,
    class_weights=class_weights_torch,
    n_epochs=30, lr=1e-3
)

print(f"\nResultat final :")
print(f"  Train : {hist_err_tr[-1]*100:.2f}%")
print(f"  Test  : {hist_err_te[-1]*100:.2f}%")
print(f"  Ecart : {abs(hist_err_tr[-1]-hist_err_te[-1])*100:.2f}%")
print(f"  Sensibilite finale (rappel malin) : {hist_sensib[-1]*100:.1f}%")

In [ ]:
# Courbes d'apprentissage du CNN medical
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4))

ax1.plot(hist_loss_tr, label="Train", color="steelblue")
ax1.plot(hist_loss_te, label="Test",  color="coral")
ax1.set_title("Loss (cross-entropy ponderee) - CNN")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot([e*100 for e in hist_err_tr], label="Train", color="steelblue")
ax2.plot([e*100 for e in hist_err_te], label="Test",  color="coral")
ax2.set_title("Taux d'erreur (%) - CNN")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Erreur (%)")
ax2.legend()
ax2.grid(alpha=0.3)

ax3.plot([s*100 for s in hist_sensib], color="darkgreen", label="Sensibilite (malin)")
ax3.axhline(y=70, color="orange",  linestyle="--", alpha=0.8, label="Seuil 70%")
ax3.axhline(y=80, color="red",     linestyle="--", alpha=0.5, label="Seuil 80%")
ax3.set_title("Sensibilite sur la classe maligne")
ax3.set_xlabel("Epoch")
ax3.set_ylabel("Sensibilite (%)")
ax3.legend()
ax3.grid(alpha=0.3)

plt.suptitle("CNN CBIS-DDSM 224x224 + augmentation - Benin vs Malin", fontsize=13)
plt.tight_layout()
plt.savefig("courbes_cnn_cbis.png", dpi=100, bbox_inches="tight")
plt.show()

---
## Section 3.4 - Analyse de la matrice de confusion et discussion médicale

En diagnostic médical, le taux d'erreur global est une métrique insuffisante.  
Il faut distinguer deux types d'erreurs aux conséquences très différentes :

| Erreur | Prediction | Realite | Consequence medicale |
|--------|-----------|---------|----------------------|
| **Faux negatif (FN)** | Benin (0) | Malin (1) | **Cancer manque → pronostic grave** |
| **Faux positif (FP)** | Malin (1) | Benin (0) | Fausse alarme → biopsie inutile, stress |

On calcule quatre métriques standard en screening :
$$\text{Sensibilite} = \frac{VP}{VP + FN} \qquad \text{Specificite} = \frac{VN}{VN + FP}$$
$$\text{VPP} = \frac{VP}{VP + FP} \qquad \text{VPN} = \frac{VN}{VN + FN}$$

In [ ]:
model_cnn.eval()
all_preds  = []
all_probas = []

with torch.no_grad():
    for Xb, yb in test_loader:
        sortie = model_cnn(Xb)
        probas = torch.softmax(sortie, dim=1)
        preds  = probas.argmax(dim=1)
        all_preds.append(preds.numpy())
        all_probas.append(probas[:, 1].numpy())   # probabilite d'etre malin

y_pred        = np.concatenate(all_preds)
y_proba_malin = np.concatenate(all_probas)

cm = confusion_matrix(y_test, y_pred)
VN = cm[0, 0];  FP = cm[0, 1]
FN = cm[1, 0];  VP = cm[1, 1]

sensibilite = VP / (VP + FN + 1e-8)
specificite = VN / (VN + FP + 1e-8)
vpp         = VP / (VP + FP + 1e-8)
vpn         = VN / (VN + FN + 1e-8)
erreur_glob = (FP + FN) / len(y_test)

print("Matrice de confusion :")
print(f"                   Predit Benin   Predit Malin")
print(f"  Reel Benin  (0) :      {VN:4d}          {FP:4d}")
print(f"  Reel Malin  (1) :      {FN:4d}          {VP:4d}")
print()
print(f"Metriques medicales :")
print(f"  Sensibilite (rappel malin) : {sensibilite*100:.2f}%  <- VP / (VP + FN)")
print(f"  Specificite (rappel benin) : {specificite*100:.2f}%  <- VN / (VN + FP)")
print(f"  VPP                        : {vpp*100:.2f}%")
print(f"  VPN                        : {vpn*100:.2f}%")
print(f"  Erreur globale             : {erreur_glob*100:.2f}%")
print(f"\nFaux negatifs (cancers manques) : {FN} sur {VP+FN} cas malins reels")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
im = ax.imshow(cm, cmap="Blues", interpolation="nearest")
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Predit Benin", "Predit Malin"], fontsize=10)
ax.set_yticklabels(["Reel Benin",   "Reel Malin"],   fontsize=10)
ax.set_xlabel("Prediction", fontsize=11)
ax.set_ylabel("Realite",    fontsize=11)
ax.set_title("Matrice de confusion - CNN 224x224 + augmentation", fontsize=11)
for i, j, etiq, coul in [(0,0,f"VN\n{VN}","#000000"),(0,1,f"FP\n{FP}","#c0392b"),
                          (1,0,f"FN\n{FN}","#c0392b"),(1,1,f"VP\n{VP}","#000000")]:
    ax.text(j, i, etiq, ha="center", va="center", fontsize=13, fontweight="bold", color=coul)

ax2     = axes[1]
noms    = ["Sensibilite\n(rappel malin)", "Specificite\n(rappel benin)", "VPP", "VPN"]
valeurs = [sensibilite*100, specificite*100, vpp*100, vpn*100]
coul    = ["#c0392b", "#2980b9", "#8e44ad", "#27ae60"]
barres  = ax2.bar(noms, valeurs, color=coul, edgecolor="white", width=0.5)
ax2.set_ylim(0, 110)
ax2.axhline(y=70, color="orange", linestyle="--", alpha=0.8, label="Seuil 70%")
ax2.axhline(y=80, color="red",    linestyle="--", alpha=0.5, label="Seuil 80%")
ax2.set_ylabel("Valeur (%)", fontsize=11)
ax2.set_title("Metriques medicales - CNN 224x224 + augmentation", fontsize=11)
ax2.legend()
ax2.grid(axis="y", alpha=0.3)
for barre, val in zip(barres, valeurs):
    ax2.text(barre.get_x() + barre.get_width()/2, val + 1,
             f"{val:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("matrice_confusion_cbis.png", dpi=100, bbox_inches="tight")
plt.show()

### Visualisation des erreurs du CNN

In [ ]:
idx_fn = np.where((y_test == 1) & (y_pred == 0))[0]
idx_fp = np.where((y_test == 0) & (y_pred == 1))[0]

n_fn_affich = min(5, len(idx_fn))
n_fp_affich = min(5, len(idx_fp))

print(f"Faux negatifs (cancers manques) : {len(idx_fn)}")
print(f"Faux positifs (fausses alarmes) : {len(idx_fp)}")

if n_fn_affich > 0 or n_fp_affich > 0:
    n_cols = max(n_fn_affich, n_fp_affich, 1)
    fig, axes = plt.subplots(2, n_cols, figsize=(3*n_cols, 6))
    if n_cols == 1:
        axes = axes.reshape(2, 1)

    for j in range(n_cols):
        if j < n_fn_affich:
            idx = idx_fn[j]
            axes[0][j].imshow(X_raw_test[idx], cmap="gray", vmin=0, vmax=1)
            axes[0][j].set_title(
                f"FN - Cancer manque\nProba malin : {y_proba_malin[idx]*100:.1f}%",
                fontsize=9, color="darkred")
        axes[0][j].axis("off")

        if j < n_fp_affich:
            idx = idx_fp[j]
            axes[1][j].imshow(X_raw_test[idx], cmap="gray", vmin=0, vmax=1)
            axes[1][j].set_title(
                f"FP - Fausse alarme\nProba malin : {y_proba_malin[idx]*100:.1f}%",
                fontsize=9, color="darkorange")
        axes[1][j].axis("off")

    plt.suptitle("Erreurs du CNN CBIS-DDSM 224x224 + augmentation", fontsize=13)
    plt.tight_layout()
    plt.savefig("erreurs_cnn_cbis.png", dpi=100, bbox_inches="tight")
    plt.show()

### Tableau comparatif final

In [ ]:
def sensibilite_dense(X_flat, y, model_type, **params):
    """
    Calcule la sensibilite (rappel malin) pour un modele dense NumPy.

    X_flat     : (n, N_PIXELS)
    y          : (n,) etiquettes binaires
    model_type : 'lin', 'h1' ou 'h2'
    params     : poids et biais selon le modele
    Retourne   : float sensibilite
    """
    if model_type == "lin":
        o = X_flat @ params["A"].T + params["b"]
    elif model_type == "h1":
        z1 = np.maximum(0, X_flat @ params["A1"].T + params["b1"])
        o  = z1 @ params["A2"].T + params["b2"]
    elif model_type == "h2":
        z1 = np.maximum(0, X_flat @ params["A1"].T + params["b1"])
        z2 = np.maximum(0, z1 @ params["A2"].T + params["b2"])
        o  = z2 @ params["A3"].T + params["b3"]
    preds = np.argmax(o, axis=1)
    mask  = (y == 1)
    vp = int((preds[mask] == 1).sum())
    fn = int((preds[mask] == 0).sum())
    return vp / (vp + fn + 1e-8)


s_lin = sensibilite_dense(X_test_flat, y_test, "lin", A=A_lin, b=b_lin)
s_h1  = sensibilite_dense(X_test_flat, y_test, "h1",  A1=A1_h1, b1=b1_h1, A2=A2_h1, b2=b2_h1)
s_h2  = sensibilite_dense(X_test_flat, y_test, "h2",  A1=A1_h2, b1=b1_h2,
                           A2=A2_h2, b2=b2_h2, A3=A3_h2, b3=b3_h2)

print("=" * 75)
print(f"{'Modele':<30} {'Params':>12} {'Train':>8} {'Test':>8} {'Sensib.':>10}")
print("=" * 75)

configs = [
    ("Lineaire",               2*N_PIXELS+2,                      etr_lin[-1], ete_lin[-1], s_lin),
    ("H=1 (128)",              128*N_PIXELS+128+2*128+2,           etr_h1[-1],  ete_h1[-1],  s_h1),
    ("H=2 (128,64)",           128*N_PIXELS+128+64*128+64+2*64+2,  etr_h2[-1],  ete_h2[-1],  s_h2),
    ("CNN 224x224 + augment.", n_params, hist_err_tr[-1], hist_err_te[-1], sensibilite),
]

for modele, params, e_tr, e_te, s in configs:
    print(f"{modele:<30} {params:>12,} {e_tr*100:>7.2f}% {e_te*100:>7.2f}% {s*100:>9.1f}%")

print("=" * 75)
print("\nNote : la sensibilite (rappel malin) est la metrique prioritaire en diagnostic.")
print("Un faux negatif = cancer non detecte = retard de diagnostic potentiellement fatal.")

### Discussion finale

#### Analyse des résultats

*(Les valeurs du tableau ci-dessus sont à reporter ici après exécution.)*

#### Pourquoi le taux d'erreur global est une métrique trompeuse ici

H=1 et H=2 affichent tous les deux ~38.89% d'erreur test — exactement le taux de cas malins dans le dataset de test (147 malins sur 378 images = 38.9%). Ce n'est pas une coïncidence : ces deux modèles prédisent **toujours "bénin"**, ce qui donne mécaniquement ce taux d'erreur. La sensibilité à 0.0% le confirme sans ambiguïté. Un tel modèle est médicalement inutilisable, quelle que soit son erreur globale.

#### Analyse modèle par modèle

**Modèle linéaire : sensibilité ~100% mais erreur test ~60%.**  
Sans pondération de la loss, le modèle linéaire a convergé vers la solution de moindre résistance : prédire systématiquement "malin". Il ne rate aucun cancer mais classe la quasi-totalité des bénins comme malins (faux positifs massifs). C'est le comportement d'un classifieur trivial biaisé vers la classe minoritaire.

**H=1 et H=2 : sensibilité 0.0% — ils ne détectent aucun cancer.**  
Les deux ont convergé vers "toujours bénin". Avec des millions de paramètres pour 1318 images, ils sont victimes de l'overfitting combiné au déséquilibre des classes. Le fait que H=1 et H=2 donnent exactement les mêmes résultats révèle que l'ajout d'une couche supplémentaire n'apporte rien : les deux ont atteint la même solution triviale.

**CNN 224×224 + augmentation : seul modèle qui apprend quelque chose.**  
Deux améliorations ont été apportées par rapport au CNN de base à 128×128 :  
la résolution 224×224 préserve les détails fins (spiculations, contours irréguliers) qui sont
les marqueurs diagnostiques clés ; la data augmentation (×4) réduit l'overfitting en exposant
le modèle à une plus grande variabilité d'orientation et de position des lésions.
La structure convolutive reste l'atout principal : les filtres 3×3 partagent leurs poids sur
toute l'image, capturant les motifs locaux quelle que soit leur position dans le crop.

#### Limites et perspectives d'amélioration

Malgré les améliorations apportées (ROI crops, 224×224, data augmentation), nos modèles restent
loin des performances cliniques attendues (sensibilité ≥ 90%). Trois facteurs fondamentaux
expliquent ce plafond.

**Le volume de données reste insuffisant.** Même avec la data augmentation qui multiplie le
dataset par 4, on reste à 5272 images artificiellement générées depuis 1318 originales.
Les modèles performants sur CBIS-DDSM dans la littérature sont tous entraînés avec du
transfer learning depuis ImageNet (1,2 million d'images) — ils bénéficient de représentations
préalablement apprises sur une immense diversité visuelle.

**La tâche est intrinsèquement difficile.** Les radiologues experts ont un taux de désaccord
de 20-30% sur les cas ambigus. Nos modèles entraînés from scratch n'ont accès qu'aux pixels
bruts, sans les connaissances anatomiques et cliniques qui guident le regard du radiologue.

**Le transfer learning reste la piste la plus prometteuse.** Fine-tuner ResNet50 ou
EfficientNet-B0 pré-entraîné sur ImageNet permettrait de contourner le problème du faible
nombre de données. Ces modèles atteignent typiquement des AUC de 0.80-0.88 sur CBIS-DDSM,
soit une sensibilité de 75-85% pour une spécificité raisonnable.